# Notebook 02: Cleaning and Feature Engineering

Inspect the cleaning pipeline and the engineered features (game_pitch_index, game_bucket, count_string).

These steps are all done inside `get_pitcher_dataset()` — this notebook shows what happens at each step.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data.fetch import load_or_fetch_pitcher
from src.data.clean import (
    select_required_columns, normalize_pitch_types,
    drop_missing_locations, sort_within_game, clean_pitcher_data
)
from src.features.pitch_index import (
    add_game_pitch_index, add_pitch_count_buckets,
    add_count_string, GAME_PITCH_INDEX_COL, GAME_BUCKET_COL
)
from src import config

pd.set_option('display.max_columns', 30)
%matplotlib inline

In [ ]:
# ---- Parameters ----
PITCHER_ID = 543037      # Gerrit Cole MLBAM ID (or use name via resolve_pitcher_id)
START_DATE = "2024-04-01"
END_DATE   = "2024-09-30"
# --------------------

## Step through cleaning

In [ ]:
raw = load_or_fetch_pitcher(PITCHER_ID, START_DATE, END_DATE)
print(f"Raw: {raw.shape[0]} rows, {raw.shape[1]} columns")

step1 = select_required_columns(raw)
print(f"After select_required_columns: {step1.shape[1]} columns")

step2 = normalize_pitch_types(step1)
print(f"Pitch types after normalization: {sorted(step2['pitch_type'].dropna().unique())}")

step3 = drop_missing_locations(step2)
print(f"After drop_missing_locations: {step3.shape[0]} rows (dropped {len(step2) - len(step3)})")

step4 = sort_within_game(step3)
print(f"Sorted. First row: {step4[['game_date','game_pk','at_bat_number','pitch_number']].iloc[0].to_dict()}")

## Feature engineering: game_pitch_index and game_bucket

In [ ]:
df = add_game_pitch_index(step4)
df = add_pitch_count_buckets(df)
df = add_count_string(df)

# Verify game_pitch_index starts at 0 for each game
assert df.groupby('game_pk')[GAME_PITCH_INDEX_COL].min().eq(0).all(), "Index should start at 0 per game"
print("✓ game_pitch_index starts at 0 per game")

print("\nBucket distribution:")
print(df[GAME_BUCKET_COL].value_counts())

df[[GAME_PITCH_INDEX_COL, GAME_BUCKET_COL, 'count_string', 'pitch_type', 'plate_x', 'plate_z']].head(10)

## Pitch index distribution per game

In [ ]:
pitches_per_game = df.groupby('game_pk')[GAME_PITCH_INDEX_COL].max() + 1

fig, ax = plt.subplots(figsize=(8, 3))
pitches_per_game.hist(ax=ax, bins=15, color='steelblue', edgecolor='black')
ax.set_xlabel("Total pitches in game")
ax.set_ylabel("# Games")
ax.set_title("Distribution of pitches per game")
plt.tight_layout()
plt.show()

print(f"Median: {pitches_per_game.median():.0f}, Min: {pitches_per_game.min()}, Max: {pitches_per_game.max()}")